## Data Preprocessing

In [16]:
import pandas as pd
from pathlib import Path

# Define the project directory (same folder as the notebook)
project_folder = Path(".")  # current folder

# Load the CSV directly from the same folder
df = pd.read_csv(project_folder / "car_data.csv", parse_dates=['DateCrawled', 'DateCreated', 'LastSeen'])

# Display general information about the dataframe
print(car_data.info())

C:\Users\Juan\AppData\Local\Temp\ipykernel_26440\2222930959.py:8: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv(project_folder / "car_data.csv", parse_dates=['DateCrawled', 'DateCreated', 'LastSeen'])
C:\Users\Juan\AppData\Local\Temp\ipykernel_26440\2222930959.py:8: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv(project_folder / "car_data.csv", parse_dates=['DateCrawled', 'DateCreated', 'LastSeen'])


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   DateCrawled        354369 non-null  datetime64[ns]
 1   Price              354369 non-null  int64         
 2   VehicleType        316879 non-null  object        
 3   RegistrationYear   354369 non-null  int64         
 4   Gearbox            334536 non-null  object        
 5   Power              354369 non-null  int64         
 6   Model              334664 non-null  object        
 7   Mileage            354369 non-null  int64         
 8   RegistrationMonth  354369 non-null  int64         
 9   FuelType           321474 non-null  object        
 10  Brand              354369 non-null  object        
 11  NotRepaired        283215 non-null  object        
 12  DateCreated        354369 non-null  datetime64[ns]
 13  NumberOfPictures   354369 non-null  int64   

In [17]:
print(df.isna().sum())

DateCrawled              0
Price                    0
VehicleType          37490
RegistrationYear         0
Gearbox              19833
Power                    0
Model                19705
Mileage                  0
RegistrationMonth        0
FuelType             32895
Brand                    0
NotRepaired          71154
DateCreated              0
NumberOfPictures         0
PostalCode               0
LastSeen                 0
dtype: int64


In [23]:
# Fill missing values for categorical features
df["VehicleType"] = df["VehicleType"].fillna("Unknown")
df["Gearbox"] = df["Gearbox"].fillna("Unknown")
df["Model"] = df["Model"].fillna("Unknown")
df["FuelType"] = df["FuelType"].fillna("Unknown")
df["NotRepaired"] = df["NotRepaired"].fillna("Unknown")

# Convert date columns to datetime with dayfirst=True
df['DateCrawled'] = pd.to_datetime(df['DateCrawled'], dayfirst=True)
df['DateCreated'] = pd.to_datetime(df['DateCreated'], dayfirst=True)
df['LastSeen'] = pd.to_datetime(df['LastSeen'], dayfirst=True)

# Create useful features
df['DaysActive'] = (df['LastSeen'] - df['DateCreated']).dt.days
df['DaysSinceCrawl'] = (df['LastSeen'] - df['DateCrawled']).dt.days

# Drop original columns that are no longer needed
df = df.drop(['DateCrawled', 'DateCreated', 'LastSeen', 'NumberOfPictures'], axis=1)

# Check for any remaining missing values
print(df.isna().sum())

Price                0
VehicleType          0
RegistrationYear     0
Gearbox              0
Power                0
Model                0
Mileage              0
RegistrationMonth    0
FuelType             0
Brand                0
NotRepaired          0
PostalCode           0
DaysActive           0
DaysSinceCrawl       0
dtype: int64


In [24]:
print(df.duplicated().sum())
df = df.drop_duplicates()

8545


In [25]:
print(df.duplicated().sum())

0


Missing values in the dataframe were filled, duplicate records were removed, and new features were created based on the date-related columns.

## Model Analysis

In [26]:
from sklearn.model_selection import train_test_split

X = df.drop("Price", axis= 1)
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12345)

In [27]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

cat_features = X.select_dtypes(include="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
    remainder="passthrough"
)

model_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

In [28]:
from sklearn.tree import DecisionTreeRegressor

model_dt = Pipeline([
    ('prep', preprocessor),
    ('model', DecisionTreeRegressor(max_depth=10, random_state=123))
])

model_dt.fit(X_train, y_train)

y_pred_dt = model_dt.predict(X_test)

In [29]:
from sklearn.ensemble import RandomForestRegressor

model_rf = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=20,
        max_depth=8,
        random_state=12345,
        n_jobs=-1
    ))
])

model_rf.fit(X_train,y_train)

y_pred_rf = model_rf.predict(X_test)

In [30]:
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder

# identificación de columnas categóricas

cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

preprocessor_lgb = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)
    ],
    remainder= "passthrough"
)


model_lgb = Pipeline([
    ("prep", preprocessor_lgb),
    ("lgb", lgb.LGBMRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=10,
    random_state=12345
))
])

model_lgb.fit(X_train,y_train)

y_pred_lgb = model_lgb.predict(X_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003414 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1051
[LightGBM] [Info] Number of data points in the train set: 276449, number of used features: 13
[LightGBM] [Info] Start training from score 4392.742741


c:\Users\Juan\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [32]:

from sklearn.metrics import mean_squared_error
import numpy as np

def rmse(y_true, y_pred):
    error = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"The root mean squared error (RMSE) of the model is: {error}")
    return error

In [33]:
import time

# For the Linear Regression model
start = time.time()
model_lr.fit(X_train, y_train)
end = time.time()
print(f"Linear Regression training time: {end - start:.4f} seconds")

Linear Regression training time: 0.8162 seconds


In [34]:
import time

# For the Decision Tree model
start = time.time()
model_dt.fit(X_train, y_train)
end = time.time()
print(f"Decision Tree training time: {end - start:.4f} seconds")

Decision Tree training time: 6.7870 seconds


In [35]:
import time

# For the Random Forest model
start = time.time()
model_rf.fit(X_train, y_train)
end = time.time()
print(f"Random Forest training time: {end - start:.4f} seconds")

Random Forest training time: 43.4412 seconds


In [36]:

import time

# For the Gradient Boosting model
start = time.time()
model_lgb.fit(X_train, y_train)
end = time.time()
print(f"Gradient Boosting training time: {end - start:.4f} seconds")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002737 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1051
[LightGBM] [Info] Number of data points in the train set: 276449, number of used features: 13
[LightGBM] [Info] Start training from score 4392.742741
Gradient Boosting training time: 1.5432 seconds


In [37]:
import time

# To calculate prediction time
start = time.time()
model_lr.predict(X_test)
end = time.time()
print(f"Linear Regression prediction time: {end - start:.4f} seconds")

Linear Regression prediction time: 0.1436 seconds


In [38]:
import time

# To calculate prediction time for the Decision Tree model
start = time.time()
model_dt.predict(X_test)
end = time.time()
print(f"Decision Tree prediction time: {end - start:.4f} seconds")

Decision Tree prediction time: 0.2233 seconds


In [39]:
import time

# To calculate prediction time for the Random Forest model
start = time.time()
model_rf.predict(X_test)
end = time.time()
print(f"Random Forest prediction time: {end - start:.4f} seconds")

Random Forest prediction time: 0.1732 seconds


In [40]:
import time

# To calculate prediction time for the Gradient Boosting model
start = time.time()
model_lgb.predict(X_test)
end = time.time()
print(f"Gradient Boosting prediction time: {end - start:.4f} seconds")

Gradient Boosting prediction time: 0.2458 seconds


c:\Users\Juan\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [41]:
# Root Mean Squared Error for Linear Regression
rmse(y_test, y_pred_lr)

The root mean squared error (RMSE) of the model is: 3411.0312392671717


np.float64(3411.0312392671717)

In [42]:
# Root Mean Squared Error for Decision Tree
rmse(y_test, y_pred_dt)

The root mean squared error (RMSE) of the model is: 2135.653819504729


np.float64(2135.653819504729)

In [43]:
# Root Mean Squared Error for Random Forest
rmse(y_test, y_pred_rf)

The root mean squared error (RMSE) of the model is: 2162.6697374163555


np.float64(2162.6697374163555)

In [ ]:
# Root Mean Squared Error for Gradient Boosting
rmse(y_test, y_pred_lgb)

| Model                        | RMSE (€) | Training Time (s) | Prediction Time (s) |
| ---------------------------- | -------- | ----------------- | ------------------- |
| Linear Regression            | 3360.89  | 1.5510            | 0.0975              |
| Decision Tree                | 2121.75  | 5.9241            | 0.0950              |
| Random Forest                | 2144.59  | 24.3259           | 0.1306              |
| Gradient Boosting (LightGBM) | 1764.51  | 4.6059            | 0.4440              |


## Conclusion

In this project, different regression models were compared to predict used car prices, evaluating both prediction quality and computational efficiency.

Linear Regression was used as a baseline model, yielding an RMSE of €3360.89, indicating limited performance due to its inability to capture non-linear relationships in the data.

Tree-based models showed a significant improvement. The Decision Tree reduced the RMSE to €2121.75, while Random Forest achieved a similar performance (€2144.59) but with a much higher computational cost in training time.

The best performance was achieved by the Gradient Boosting model (LightGBM), with an RMSE of €1764.51, representing a substantial improvement over the other models. Its training time was relatively low compared to Random Forest, although its prediction time was the highest.

These results indicate that LightGBM provides the best balance between accuracy and efficiency for this problem. Its ability to model complex relationships and handle categorical variables makes it the most suitable choice for estimating the market value of vehicles in this context.

Finally, it is observed that the price distribution is skewed towards high values (outliers), which impacts the RMSE metric. Nevertheless, boosting models handle this complexity better compared to simpler models.